In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from NOTEBOOKS.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [25]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 1


### Load Player Data and Bookmaker Data

In [26]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

### Update projected starting lineups

In [27]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Applications/Documents/NBA-Prop-Predictor/MODELS/teamInfo.py
Updated 12 teams with confirmed lineups


### Top EVs for single bets

In [12]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 250) & (usData['ODDS'] >= -250)]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                             edge_threshold=0.20, stake=10, 
                             variance_inflation=1.1, distribution_type='skewnorm', 
                             use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']].head(10)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,EXPECTED ROI,KELLY_FRACTION,SIGMA FLAG
0,Nikola Jokic,Bovada,22.5,21.25,Under,200,1,6.75,67.5,0.337,High
1,Chet Holmgren,Bovada,20.5,21.61,Over,200,1,6.58,65.8,0.329,High
2,Chet Holmgren,Bovada,19.5,21.61,Over,160,1,5.69,56.9,0.355,High
3,Nikola Jokic,Bovada,23.5,21.25,Under,160,1,5.59,55.9,0.349,High
4,Isaiah Hartenstein,Bovada,13.5,14.09,Over,190,0,5.46,54.6,0.288,High


## Top EVs for 2 leg bets

### Underdog picks

In [28]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=10, 
                     variance_inflation=1.1, distribution_type='skewnorm',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Lonzo Ball,Alex Sarr,9.5,16.5,under,over,1,4.19,0.210,Low,High
1,Duncan Robinson,Alex Sarr,13.5,16.5,under,over,1,4.18,0.209,Med,High
2,Caris LeVert,Alex Sarr,10.5,16.5,under,over,1,4.16,0.208,Med,High
3,Duncan Robinson,Lonzo Ball,13.5,9.5,under,under,1,3.86,0.193,Med,Low
4,Caris LeVert,Lonzo Ball,10.5,9.5,under,under,1,3.84,0.192,Med,Low


### Prizepicks picks

In [29]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=10, 
                     variance_inflation=1.1, distribution_type='skewnorm',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Lonzo Ball,Alex Sarr,9.5,16.5,under,over,1,4.19,0.210,Low,High
1,Aaron Nesmith,Alex Sarr,17.5,16.5,under,over,1,3.85,0.192,High,High
2,Chet Holmgren,Alex Sarr,17.5,16.5,over,over,1,3.71,0.185,High,High
3,Dean Wade,Alex Sarr,8.5,16.5,under,over,1,3.62,0.181,Low,High
4,Lonzo Ball,Kyshawn George,9.5,12.5,under,over,1,3.61,0.180,Low,High


## 3 leg parlay

### Underdog picks

In [30]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.10, stake=10, 
                     variance_inflation=1.1, distribution_type='skewnorm', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3','MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Duncan Robinson,Lonzo Ball,Alex Sarr,13.5,9.5,16.5,under,under,over,1,8.29,0.166,Med,Low,High
1,Caris LeVert,Lonzo Ball,Alex Sarr,10.5,9.5,16.5,under,under,over,1,8.27,0.165,Med,Low,High
2,Caris LeVert,Duncan Robinson,Alex Sarr,10.5,13.5,16.5,under,under,over,1,8.25,0.165,Med,Med,High
3,Caris LeVert,Duncan Robinson,Lonzo Ball,10.5,13.5,9.5,under,under,under,1,7.84,0.157,Med,Med,Low
4,Lonzo Ball,Dean Wade,Alex Sarr,9.5,8.5,16.5,under,under,over,1,7.58,0.152,Low,Low,High


### Prizepicks picks

In [31]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.40, stake=100, 
                     variance_inflation=1.1, distribution_type='skewnorm', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Duncan Robinson,Lonzo Ball,Alex Sarr,13.5,9.5,16.5,under,under,over,1,8.29,0.166,Med,Low,High
1,Caris LeVert,Lonzo Ball,Alex Sarr,10.5,9.5,16.5,under,under,over,1,8.27,0.165,Med,Low,High
2,Caris LeVert,Duncan Robinson,Alex Sarr,10.5,13.5,16.5,under,under,over,1,8.25,0.165,Med,Med,High
3,Caris LeVert,Duncan Robinson,Lonzo Ball,10.5,13.5,9.5,under,under,under,1,7.84,0.157,Med,Med,Low
4,Lonzo Ball,Dean Wade,Alex Sarr,9.5,8.5,16.5,under,under,over,1,7.58,0.152,Low,Low,High


In [17]:
def count_line_hits(player_df, line, category, game_windows=[5, 10, 15]):
    results = {}
    player_df_sorted = player_df.sort_values('GAME_DATE')
    total_games = len(player_df_sorted)

    for window in game_windows:
        # Handle players with fewer games
        if total_games < window:
            last_n_games = player_df_sorted
        else:
            last_n_games = player_df_sorted.tail(window)

        if category == 'player_points':
            hits = (last_n_games['PTS'] > line).sum()
        elif category == 'player_assists':
            hits = (last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds':
            hits = (last_n_games['REB'] > line).sum()
        elif category == 'player_threes':
            hits = (last_n_games['FG3M'] > line).sum()
        elif category == 'player_blocks':
            hits = (last_n_games['BLK'] > line).sum()
        elif category == 'player_steals':
            hits = (last_n_games['STL'] > line).sum()
        elif category == 'player_field_goals':
            hits = (last_n_games['FGM'] > line).sum()
        elif category == 'player_frees_made':
            hits = (last_n_games['FTM'] > line).sum()
        elif category == 'player_points_rebounds_assists':
            hits = (last_n_games['PTS'] + last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_points_rebounds':
            hits = (last_n_games['PTS'] + last_n_games['REB'] > line).sum()
        elif category == 'player_points_assists':
            hits = (last_n_games['PTS'] + last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds_assists':
            hits = (last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_turnovers':
            hits = (last_n_games['TOV'] > line).sum()
        else:
            hits = 0

        results['NAME'] = player_df_sorted['PLAYER_NAME'].iloc[0] if total_games > 0 else 'Unknown'
        results['CATEGORY'] = category
        results['LINE'] = line
        results[f'L-{window}'] = round(hits / window, 2)


    return results

prizePicks = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks')]
prizePicks= prizePicks.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
prizePicks

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
118,PrizePicks,player_points,Michael Porter Jr,Over,22.5,-137,2025-11-12,2025-11-11T18:25:36Z
120,PrizePicks,player_points,Brandon Ingram,Over,20.5,-137,2025-11-12,2025-11-11T18:25:36Z
122,PrizePicks,player_points,R.J. Barrett,Over,20.5,-137,2025-11-12,2025-11-11T18:25:36Z
124,PrizePicks,player_points,Scottie Barnes,Over,18.5,-137,2025-11-12,2025-11-11T18:25:36Z
126,PrizePicks,player_points,Immanuel Quickley,Over,15.5,-137,2025-11-12,2025-11-11T18:25:36Z
...,...,...,...,...,...,...,...,...
2292,PrizePicks,player_blocks_steals,Isaiah Hartenstein,Over,1.5,-137,2025-11-12,2025-11-11T18:26:56Z
2294,PrizePicks,player_blocks_steals,Draymond Green,Over,1.5,-137,2025-11-12,2025-11-11T18:26:56Z
2296,PrizePicks,player_blocks_steals,Pascal Siakam,Over,1.5,-137,2025-11-12,2025-11-11T18:26:50Z
2298,PrizePicks,player_blocks_steals,Isaiah Collier,Over,0.5,-137,2025-11-12,2025-11-11T18:26:50Z


In [18]:
line_hit_data = []

for index, row in prizePicks.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = s26[s26['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

        
        

Saved 83 records for player_points to player_points.csv
Saved 52 records for player_rebounds to player_rebounds.csv
Saved 31 records for player_assists to player_assists.csv
Saved 8 records for player_threes to player_threes.csv
Saved 4 records for player_blocks to player_blocks.csv
Saved 16 records for player_steals to player_steals.csv
Saved 19 records for player_field_goals to player_field_goals.csv
Saved 17 records for player_frees_made to player_frees_made.csv
Saved 81 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 74 records for player_points_rebounds to player_points_rebounds.csv
Saved 74 records for player_points_assists to player_points_assists.csv
Saved 50 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 20 records for player_turnovers to player_turnovers.csv
Saved 13 records for player_blocks_steals to player_blocks_steals.csv

All category files saved to ../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS


In [19]:
underdog = dfsData[(dfsData['BOOKMAKER'] == 'Underdog')]
underdog= underdog.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
line_hit_data = []

for index, row in underdog.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = s26[s26['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

Saved 50 records for player_points to player_points.csv
Saved 14 records for player_rebounds to player_rebounds.csv
Saved 12 records for player_assists to player_assists.csv
Saved 9 records for player_threes to player_threes.csv
Saved 1 records for player_blocks to player_blocks.csv
Saved 1 records for player_steals to player_steals.csv
Saved 2 records for player_frees_made to player_frees_made.csv
Saved 73 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 27 records for player_points_rebounds to player_points_rebounds.csv
Saved 22 records for player_points_assists to player_points_assists.csv
Saved 17 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 4 records for player_turnovers to player_turnovers.csv
Saved 1 records for player_blocks_steals to player_blocks_steals.csv

All category files saved to ../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG
